In [0]:
df_order_items = spark.table("`e-commerce_sales_catalog`.bronze.bronze_order_items")

In [0]:
df_order_items.printSchema()
print("Initial Count:", df_order_items.count())

In [0]:
df_order_items = df_order_items.dropDuplicates(["order_id", "order_item_id"])

In [0]:
from pyspark.sql.functions import col, sum

df_order_items.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_order_items.columns
]).show()

In [0]:
from pyspark.sql.functions import coalesce, lit

df_order_items = df_order_items.withColumn(
    "price",
    coalesce(col("price"), lit("0"))
)

df_order_items = df_order_items.withColumn(
    "freight_value",
    coalesce(col("freight_value"), lit("0"))
)

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_order_items = df_order_items.withColumn("price", col("price").cast("double")) \
                               .withColumn("freight_value", col("freight_value").cast("double")) \
                               .withColumn("order_item_id", col("order_item_id").cast("int")) \
                               .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))

In [0]:
df_order_items.display()
df_order_items.printSchema()
print("Final Count:", df_order_items.count())

In [0]:
df_order_items.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.silver.silver_order_items")